In [ ]:
pip install pandas yahoo_fin

In [ ]:
pip install pandas numpy investpy

In [ ]:
!pip install yfinance pandas requests

In [ ]:
# (한 번만) 필요한 패키지 설치
!pip install --upgrade yfinance pandas

In [ ]:
!pip install pandas_datareader


In [ ]:
# ────────────────────────────────────────────────────────────────
# 1) 라이브러리 설치 (한 번만 실행)
%pip install finvizfinance


In [ ]:
%pip install finviz


In [ ]:
%pip install finvizfinance

# MACRO 가장 최근 가격 반영

### CPI, west Texas gas price
### VIX index
### Long / Short interest rate
### Employment Rate
### High Yiedl Spread Adjusted



In [ ]:
# ────────────────────────────────────────────────────────────────
# (실행 전 커널 재시작 권장)
import os
from datetime import datetime, timedelta
import pandas as pd
from pandas_datareader import data as pdr
import yfinance as yf

# ────────────────────────────────────────────────────────────────
# MACRO_DIR = r"C:\Users\LabPC\OneDrive\주식\Macro Data"
MACRO_DIR = r"C:\Users\LabPC\OneDrive\주식\Macro Data"
os.makedirs(MACRO_DIR, exist_ok=True)

INDICATORS = {
    "CPI":           ("fred",   "CPIAUCSL"),
    "WTI":           ("fred",   "MCOILWTICO"),
    "VIX":           ("fred",   "VIXCLS"),
    "FedFundsRate":      ("fred",   "FEDFUNDS"),
    "YieldCurve":    ("fred",  ("GS10","GS2")),
    "Unemployment":  ("fred",   "UNRATE"),
    "HY_Spread":     ("fred",   "BAMLH0A3HYCEY"),
}

def fetch_fred(series, start, end):
    return pdr.DataReader(series, "fred", start, end)

def fetch_yahoo(symbol, start, end):
    df = yf.download(
        symbol,
        start=start.strftime("%Y-%m-%d"),
        end  =end  .strftime("%Y-%m-%d"),
        progress=False
    )
    return df[["Close"]].rename(columns={"Close":"Value"})

for name, (source, code) in INDICATORS.items():
    print(f"\n▶ {name} 업데이트 시작")
    csv_old = None

    # 기존 파일 찾기
    for fn in os.listdir(MACRO_DIR):
        if fn.endswith(f" {name}.csv"):
            csv_old = os.path.join(MACRO_DIR, fn)
            break

    # 1) 첫 생성인지
    if csv_old is None:
        print("  – 초기 파일 생성")
        start = datetime(1970,1,1)
    else:
        df_old = pd.read_csv(csv_old, parse_dates=["Date"])
        start   = df_old["Date"].max() + timedelta(days=1)

    end = datetime.today() + timedelta(days=1)

    # 2) 새로운 데이터 가져오기 (실패해도 df는 빈 DF)
    try:
        if source == "fred":
            if isinstance(code, tuple):
                df1 = fetch_fred(code[0], start, end)
                df2 = fetch_fred(code[1], start, end)
                df = pd.DataFrame({
                    "Date": df1.index,
                    "Value": df1[code[0]] - df2[code[1]]
                })
            else:
                df = fetch_fred(code, start, end).reset_index()
                df.columns = ["Date","Value"]
        else:  # yahoo
            df = fetch_yahoo(code, start, end).reset_index()
            df.columns = ["Date","Value"]
    except Exception as e:
        print(f"  ✖ {name} 데이터 fetch 실패: {e}")
        df = pd.DataFrame(columns=["Date","Value"])

    if df.empty:
        print("  – 신규 데이터 없음, 기존 데이터만으로 정렬·저장")

    # 3) 합치고 중복 제거 후 내림차순 정렬
    if csv_old is None:
        combined = df.copy()
    else:
        combined = pd.concat([df_old, df], ignore_index=True)

    combined = (
        combined
        .drop_duplicates(subset="Date", keep="first")
        .sort_values("Date", ascending=False)
        .reset_index(drop=True)
    )

    # CPI만 YoY 계산
    if name == "CPI":
        asc = combined.sort_values("Date").reset_index(drop=True)
        asc["YoY_%"] = (asc["Value"].pct_change(12) * 100).round(2)
        combined = asc.sort_values("Date", ascending=False).reset_index(drop=True)

    # 4) 파일명에 기간 붙이고 저장
    start_str = combined["Date"].min().strftime("%Y.%m.%d")
    end_str   = combined["Date"].max().strftime("%Y.%m.%d")
    new_fn    = f"{start_str}_{end_str} {name}.csv"
    new_path  = os.path.join(MACRO_DIR, new_fn)

    combined.to_csv(new_path, index=False, date_format="%Y-%m-%d")
    print(f"  ✔ 저장: {new_fn}")

    # 5) 이전 파일 삭제
    if csv_old and new_path != csv_old:
        os.remove(csv_old)


# Stock들 가장 최근 가격 반영

### 너가 원하는 티커를 적어 아래에

In [ ]:
# # (이 셀 실행 전에 커널 Restart 권장)
# import os, time, requests
# from datetime import datetime, timedelta
# import pandas as pd

# # ROOT_DIR = r"C:\Users\LabPC\OneDrive\주식\Back Test"
# ROOT_DIR = r"C:\Users\LabPC\OneDrive\주식\Back Test"

# TICKER_MAP = {
#     "apple":      "AAPL",
#     "bbh":        "BBAI",
#     "bitcoin":    "BTC-USD",
#     "coupang":    "CPNG",
#     "crispr":     "CRSP",
#     "google":     "GOOGL",
#     "grail":      "GRAL",
#     "illumina":   "ILMN",
#     "nvidia":     "NVDA",
#     "occidental": "OXY",
#     "snowflake":  "SNOW",
#     "tesla":      "TSLA",
#     "unity":      "U",
#     "xrp":        "XRP-USD",
# }

# HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

# def to_epoch(dt: datetime) -> int:
#     return int(time.mktime(dt.timetuple()))

# def format_vol(x):
#     if pd.isna(x): return ""
#     v = int(x)
#     if v >= 1_000_000: return f"{v/1_000_000:.2f}M"
#     if v >=   1_000:   return f"{v/1_000:.2f}K"
#     return str(v)

# def fetch_chart_json(ticker, start_dt, end_dt):
#     """Yahoo chart API로 raw JSON 받아와 DataFrame 리턴"""
#     p1, p2 = to_epoch(start_dt), to_epoch(end_dt)
#     url = (
#         f"https://query1.finance.yahoo.com/v8/finance/chart/{ticker}"
#         f"?symbol={ticker}&period1={p1}&period2={p2}"
#         "&interval=1d&includePrePost=false"
#     )
#     r = requests.get(url, headers=HEADERS)
#     data = r.json()
#     if "chart" not in data or not data["chart"]["result"]:
#         return pd.DataFrame()
#     res    = data["chart"]["result"][0]
#     ts     = res["timestamp"]
#     quote  = res["indicators"]["quote"][0]
#     df = pd.DataFrame({
#         "Date":   [datetime.fromtimestamp(t) for t in ts],
#         "Open":   quote["open"],
#         "High":   quote["high"],
#         "Low":    quote["low"],
#         "Price":  quote["close"],
#         "Volume": quote["volume"]
#     })
#     return df

# # ────────────────────────────────────────────────────────────────────
# # 1) 매핑만 있고 폴더가 없는 종목은 폴더 생성
# # 2) 폴더에 CSV 0개면 전체 히스토리(1970~오늘) 로 드롭다운
# for comp_lower, ticker in TICKER_MAP.items():
#     comp_name = comp_lower.capitalize()
#     folder = os.path.join(ROOT_DIR, comp_name)
#     os.makedirs(folder, exist_ok=True)

#     csvs = [f for f in os.listdir(folder) if f.lower().endswith(".csv")]
#     if not csvs:
#         print(f"▶ {comp_name}: 초기 CSV 생성 중…")
#         df_init = fetch_chart_json(ticker, datetime(1970,1,1), datetime.today()+timedelta(days=1))
#         if df_init.empty:
#             print(f"  ✖ {ticker} 데이터가 하나도 없습니다.")
#             continue
#         # 포맷 정리
#         df_init["Vol."]     = df_init["Volume"].apply(format_vol)
#         df_init["Change %"] = (df_init["Price"].pct_change()*100)\
#                               .map(lambda x: f"{x:.2f}%")
#         df_init = df_init[["Date","Price","Open","High","Low","Vol.","Change %"]]
#         # 문자열 날짜
#         df_init["Date"] = df_init["Date"].dt.strftime("%m/%d/%Y")
#         fn = f"{comp_name} Historical Data.csv"
#         df_init.to_csv(os.path.join(folder, fn), index=False)
#         print(f"  ✔ {fn} 생성 완료")
# # ────────────────────────────────────────────────────────────────────
# # 3) 기존 업데이트 로직
# for company in os.listdir(ROOT_DIR):
#     folder = os.path.join(ROOT_DIR, company)
#     if not os.path.isdir(folder): continue
#     ticker = TICKER_MAP.get(company.lower(), None)
#     if ticker is None:
#         continue

#     for fn in os.listdir(folder):
#         if not fn.lower().endswith(".csv"): continue

#         old_path = os.path.join(folder, fn)
#         print(f"\n▶ {company} ({ticker}) 업데이트 중: {fn}")

#         # 기존 읽기
#         df_old = pd.read_csv(old_path, parse_dates=["Date"])
#         last   = df_old["Date"].max()
#         start  = last + timedelta(days=1)
#         end    = datetime.today() + timedelta(days=1)

#         # 신규 데이터 받기
#         df_app = pd.DataFrame()
#         if start.date() < end.date():
#             df_new = fetch_chart_json(ticker, start, end)
#             if not df_new.empty:
#                 df_new["Vol."]     = df_new["Volume"].apply(format_vol)
#                 df_new["Change %"] = (df_new["Price"].pct_change()*100)\
#                                       .map(lambda x: f"{x:.2f}%")
#                 df_app = df_new[["Date","Price","Open","High","Low","Vol.","Change %"]]
#                 print(f"  ✔ {len(df_app)}개 행 신규 추가")
#             else:
#                 print("  – 신규 데이터 없음 또는 API 결과 비어있음")
#         else:
#             print("  – 신규 데이터 없음")

#         # 합치고 중복 제거 & 최신순 정렬
#         df_combined = pd.concat([df_old, df_app], ignore_index=True)
#         df_combined = df_combined.drop_duplicates(subset="Date", keep="first")
#         df_combined = df_combined.sort_values("Date", ascending=False) \
#                                  .reset_index(drop=True)

#         # 날짜 범위, 문자열 포맷
#         min_d = df_combined["Date"].min().strftime("%Y.%m.%d")
#         max_d = df_combined["Date"].max().strftime("%Y.%m.%d")
#         df_combined["Date"] = df_combined["Date"].dt.strftime("%m/%d/%Y")

#         # 새 파일명 & 저장
#         new_fn = f"{min_d}_{max_d} {fn}"
#         new_path = os.path.join(folder, new_fn)
#         df_combined.to_csv(new_path, index=False)

#         # 옛 파일 지우기
#         if new_path != old_path:
#             os.remove(old_path)

#         print(f"  ✔ 저장 완료: {new_fn} (최신순 정렬)")


# Stock들 가장 최근 가격 반영

### 너가 원하는 티커를 적어 아래에 (V2)

In [ ]:
# # (이 셀 실행 전에 커널 Restart 권장)
# import os, time, requests
# from datetime import datetime, timedelta
# import pandas as pd

# # ROOT_DIRECTORY 설정
# # 필요에 따라 경로를 수정하세요
# ROOT_DIR = r"C:\Users\LabPC\OneDrive\주식\Back Test"

# # 종목별 티커 매핑
# TICKER_MAP = {
#     "apple":      "AAPL",
#     "bbh":        "BBAI",
#     "bitcoin":    "BTC-USD",
#     "coupang":    "CPNG",
#     "crispr":     "CRSP",
#     "google":     "GOOGL",
#     "grail":      "GRAL",
#     "illumina":   "ILMN",
#     "nvidia":     "NVDA",
#     "occidental": "OXY",
#     "snowflake":  "SNOW",
#     "tesla":      "TSLA",
#     "unity":      "U",
#     "xrp":        "XRP-USD",
# }

# # HTTP 요청 헤더
# HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}


# def to_epoch(dt: datetime) -> int:
#     return int(time.mktime(dt.timetuple()))


# def format_vol(x):
#     if pd.isna(x): return ""
#     v = int(x)
#     if v >= 1_000_000: return f"{v/1_000_000:.2f}M"
#     if v >=   1_000:   return f"{v/1_000:.2f}K"
#     return str(v)


# def fetch_chart_json(ticker, start_dt, end_dt):
#     p1, p2 = to_epoch(start_dt), to_epoch(end_dt)
#     url = (
#         f"https://query1.finance.yahoo.com/v8/finance/chart/{ticker}"
#         f"?period1={p1}&period2={p2}&interval=1d&includePrePost=false"
#     )
#     r = requests.get(url, headers=HEADERS)
#     data = r.json()
#     if "chart" not in data or not data["chart"]["result"]:
#         return pd.DataFrame()
#     res = data["chart"]["result"][0]
#     ts, q = res["timestamp"], res["indicators"]["quote"][0]
#     return pd.DataFrame({
#         "Date":  [datetime.fromtimestamp(t) for t in ts],
#         "Open":  q["open"],
#         "High":  q["high"],
#         "Low":   q["low"],
#         "Price": q["close"],
#         "Volume":q["volume"],
#     })

# # ────────────────────────────────────────────────────────────────────
# # 1) 폴더 생성 및 초기 CSV (전체 히스토리)
# # ────────────────────────────────────────────────────────────────────
# for comp, ticker in TICKER_MAP.items():
#     name = comp.capitalize()
#     folder = os.path.join(ROOT_DIR, name)
#     os.makedirs(folder, exist_ok=True)
#     # 'Historical Data.csv' 파일이 없으면 전체 데이터 저장
#     base_fn = f"{name} Historical Data.csv"
#     base_path = os.path.join(folder, base_fn)
#     if not os.path.exists(base_path):
#         print(f"▶ {name}: 초기 CSV 생성 중…")
#         df_all = fetch_chart_json(ticker, datetime(1970,1,1), datetime.today()+timedelta(days=1))
#         if df_all.empty:
#             print(f"  ✖ {ticker} 데이터 없음")
#             continue
#         df_all["Vol."]     = df_all["Volume"].apply(format_vol)
#         df_all["Change %"] = (df_all["Price"].pct_change()*100).map(lambda x: f"{x:.2f}%")
#         df_all = df_all[["Date","Price","Open","High","Low","Vol.","Change %"]]
#         df_all["Date"] = df_all["Date"].dt.strftime("%m/%d/%Y")
#         df_all.to_csv(base_path, index=False)
#         print(f"  ✔ {base_fn} 생성 완료")

# # ────────────────────────────────────────────────────────────────────
# # 2) 기존 'Historical Data.csv' 업데이트 및 파일명 재생성
# # ────────────────────────────────────────────────────────────────────
# for comp, ticker in TICKER_MAP.items():
#     name = comp.capitalize()
#     folder = os.path.join(ROOT_DIR, name)
#     base_fn = f"{name} Historical Data.csv"
#     base_path = os.path.join(folder, base_fn)
#     if not os.path.exists(base_path):
#         continue

#     print(f"\n▶ {name} ({ticker}) 업데이트 중…")
#     df_old = pd.read_csv(base_path, parse_dates=["Date"] )
#     last = df_old["Date"].max()
#     start = last + timedelta(days=1)
#     end = datetime.today() + timedelta(days=1)

#     df_new = pd.DataFrame()
#     if start.date() < end.date():
#         tmp = fetch_chart_json(ticker, start, end)
#         if not tmp.empty:
#             tmp["Vol."]     = tmp["Volume"].apply(format_vol)
#             tmp["Change %"] = (tmp["Price"].pct_change()*100).map(lambda x: f"{x:.2f}%")
#             df_new = tmp[["Date","Price","Open","High","Low","Vol.","Change %"]]
#             print(f"  ✔ {len(df_new)}개 신규 행 추가")
#         else:
#             print("  – 신규 데이터 없음")
#     else:
#         print("  – 신규 데이터 없음")

#     # 병합 및 정리
#     df_combined = pd.concat([df_old, df_new], ignore_index=True)
#     df_combined = df_combined.drop_duplicates(subset="Date").sort_values("Date", ascending=False)

#     # 날짜 범위와 포맷 설정
#     max_d = df_combined["Date"].max().strftime("%Y.%m.%d")
#     min_d = df_combined["Date"].min().strftime("%Y.%m.%d")
#     df_combined["Date"] = df_combined["Date"].dt.strftime("%m/%d/%Y")

#     # 새 파일명 생성 및 저장
#     new_fn = f"{max_d}-{min_d} {name} Historical Data.csv"
#     new_path = os.path.join(folder, new_fn)
#     df_combined.to_csv(new_path, index=False)

#     # 이전 CSV 파일 제거, 새 파일만 남김
#     for f in os.listdir(folder):
#         if f.endswith('.csv') and f != new_fn:
#             os.remove(os.path.join(folder, f))

#     print(f"  ✔ 저장 완료: {new_fn}")


In [ ]:
import os, time, requests
from datetime import datetime, timedelta
import pandas as pd

# 🔹 티커 ↔ 회사명 매핑
ticker_names = {
    "GOOG": "alphabet",
    "BN": "brookfield",
    "OXY": "occidental-petroleum",
    "CRSP": "crispr-therapeutics-ag",
    "NVO": "novo-nordisk",
    "PATH": "uipath",
    "OKTA": "okta",
    "ILMN": "illumina",
    "TSM": "taiwan-semiconductor-manufacturing",
    "CRCL": "circle-internet",
    "NTRA": "natera",
    "META": "meta-platforms",
    "NU": "nu-holdings",
    "CPNG": "coupang",
    "CVX": "chevron",
    "PLTR": "palantir-technologies",
    "DDOG": "datadog",
    "CRWD": "crowdstrike",
    "AVGO": "broadcom",
    "UNH": "unitedhealth-group",
    "LMND": "lemonade",
    "AAPL": "apple",
    "MITK": "mitek-systems",
    "MSFT": "microsoft",
    "BBAI": "bigbearai-holdings",
    "VST": "vistra",
    "VRT": "vertiv-holdings",
    "MP": "mp-materials",
    "COST": "costco",
    "TEVA": "teva-pharmaceutical-industries",
    "TSLA": "tesla",
    "U": "unity-software",
    "RDDT": "reddit",
    "SNOW": "snowflake",
    "RKLB": "rocket-lab",
    "MELI": "mercadolibre",
    "EH": "ehang-holdings",
    "GRAIL": "grail",
    "PLUG": "plug-power",
    "SE": "sea",
    "TEM": "tempus-ai",
    "IREN": "iren",
    "PI": "impinj",
    "JOBY": "joby-aviation",
    "APP": "applovin",
    "COIN": "coinbase-gloabl",
    "BTC-USD": "bitcoin",
    "XRP-USD": "xrp",
    "NVDA":"nvidia"
}

# 📁 저장 경로 설정
ROOT_DIR = r"C:\Users\LabPC\OneDrive\주식\Back Test"
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

def to_epoch(dt: datetime) -> int:
    return int(time.mktime(dt.timetuple()))

def format_vol(x):
    if pd.isna(x): return ""
    v = int(x)
    if v >= 1_000_000: return f"{v/1_000_000:.2f}M"
    if v >=   1_000:   return f"{v/1_000:.2f}K"
    return str(v)

def fetch_chart_json(ticker, start_dt, end_dt):
    p1, p2 = to_epoch(start_dt), to_epoch(end_dt)
    url = (
        f"https://query1.finance.yahoo.com/v8/finance/chart/{ticker}"
        f"?period1={p1}&period2={p2}&interval=1d&includePrePost=false"
    )
    r = requests.get(url, headers=HEADERS)
    data = r.json()
    if "chart" not in data or not data["chart"]["result"]:
        return pd.DataFrame()
    res = data["chart"]["result"][0]
    ts, q = res["timestamp"], res["indicators"]["quote"][0]
    return pd.DataFrame({
        "Date":  [datetime.fromtimestamp(t) for t in ts],
        "Open":  q["open"],
        "High":  q["high"],
        "Low":   q["low"],
        "Price": q["close"],
        "Volume":q["volume"],
    })

# ───────────────────────────────────────────────────────────
# 1) 폴더 생성 및 초기 CSV (전체 히스토리)
# ───────────────────────────────────────────────────────────
for ticker, company in ticker_names.items():
    name = company.replace("-", " ").title()
    folder = os.path.join(ROOT_DIR, name)
    os.makedirs(folder, exist_ok=True)

    base_fn = f"{name} Historical Data.csv"
    base_path = os.path.join(folder, base_fn)

    if not os.path.exists(base_path):
        print(f"▶ {name}: 초기 CSV 생성 중…")
        df_all = fetch_chart_json(ticker, datetime(1970,1,1), datetime.today()+timedelta(days=1))
        if df_all.empty:
            print(f"  ✖ {ticker} 데이터 없음")
            continue
        df_all["Vol."]     = df_all["Volume"].apply(format_vol)
        df_all["Change %"] = (df_all["Price"].pct_change()*100).map(lambda x: f"{x:.2f}%")
        df_all = df_all[["Date","Price","Open","High","Low","Vol.","Change %"]]
        df_all["Date"] = df_all["Date"].dt.strftime("%m/%d/%Y")
        df_all.to_csv(base_path, index=False)
        print(f"  ✔ {base_fn} 생성 완료")

# ───────────────────────────────────────────────────────────
# 2) 기존 CSV 업데이트 및 파일명 재생성
# ───────────────────────────────────────────────────────────
for ticker, company in ticker_names.items():
    name = company.replace("-", " ").title()
    folder = os.path.join(ROOT_DIR, name)

    base_fn = f"{name} Historical Data.csv"
    base_path = os.path.join(folder, base_fn)
    if not os.path.exists(base_path):
        continue

    print(f"\n▶ {name} ({ticker}) 업데이트 중…")
    df_old = pd.read_csv(base_path, parse_dates=["Date"] )
    last = df_old["Date"].max()
    start = last + timedelta(days=1)
    end = datetime.today() + timedelta(days=1)

    df_new = pd.DataFrame()
    if start.date() < end.date():
        tmp = fetch_chart_json(ticker, start, end)
        if not tmp.empty:
            tmp["Vol."]     = tmp["Volume"].apply(format_vol)
            tmp["Change %"] = (tmp["Price"].pct_change()*100).map(lambda x: f"{x:.2f}%")
            df_new = tmp[["Date","Price","Open","High","Low","Vol.","Change %"]]
            print(f"  ✔ {len(df_new)}개 신규 행 추가")
        else:
            print("  – 신규 데이터 없음")
    else:
        print("  – 신규 데이터 없음")

    # 병합 및 정리
    df_combined = pd.concat([df_old, df_new], ignore_index=True)
    df_combined = df_combined.drop_duplicates(subset="Date").sort_values("Date", ascending=False)

    # 날짜 범위와 포맷 설정
    max_d = df_combined["Date"].max().strftime("%Y.%m.%d")
    min_d = df_combined["Date"].min().strftime("%Y.%m.%d")
    df_combined["Date"] = df_combined["Date"].dt.strftime("%m/%d/%Y")

    # 새 파일명 생성 및 저장
    new_fn = f"{max_d}-{min_d} {name} Historical Data.csv"
    new_path = os.path.join(folder, new_fn)
    df_combined.to_csv(new_path, index=False)

    # 이전 CSV 파일 제거, 새 파일만 남김
    for f in os.listdir(folder):
        if f.endswith('.csv') and f != new_fn:
            os.remove(os.path.join(folder, f))

    print(f"  ✔ 저장 완료: {new_fn}")


# 위의 티커들 Financial Data

### Quarter 마다, Yearly
### EBITA / ROI / EPS / PEG grwoth


In [ ]:
import pandas as pd
import os

# ─────────────────────────────────────────────────────────────
# 사용자 설정: 경로와 티커 리스트
# ─────────────────────────────────────────────────────────────
# base_folder = r"C:/Users/LabPC/OneDrive/주식/Financial Data"  # ← 여기를 본인 경로로 바꿔주세요 (역슬래시 대신 슬래시)
base_folder = r"C:\Users\LabPC\OneDrive\주식/Financial Data"  # ← 여기를 본인 경로로 바꿔주세요 (역슬래시 대신 슬래시)
tickers = {
    "AAPL": 320193,
    # "TSLA": 1318605,  # 추가 가능
}

tags_needed = {
    "Revenue": ["Revenues", "RevenueFromContractWithCustomerExcludingAssessedTax"],
    "NetIncome": ["NetIncomeLoss"],
    "TotalAssets": ["Assets"],
    "TotalEquity": ["StockholdersEquity", "StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest"],
    "TotalLiabilities": ["Liabilities"],
    "OperatingCashFlow": ["NetCashProvidedByUsedInOperatingActivities"],
    "CapEx": ["PaymentsToAcquirePropertyPlantAndEquipment"],
    "EBITDA": ["OperatingIncomeLoss"],
    "SharesOutstanding": ["WeightedAverageNumberOfDilutedSharesOutstanding"],
    "EPS": ["EarningsPerShareDiluted"],
    "TotalDebt": ["LongTermDebt", "Debt"],
}

quarters = sorted([f for f in os.listdir(base_folder) if f.startswith("20") and os.path.isdir(os.path.join(base_folder, f))])
stock_root = os.path.join(base_folder, "Stocks")
os.makedirs(stock_root, exist_ok=True)

for ticker, cik in tickers.items():
    all_data = []

    for quarter in quarters:
        quarter_path = os.path.join(base_folder, quarter)
        sub_path = os.path.join(quarter_path, "sub.txt")
        num_path = os.path.join(quarter_path, "num.txt")

        if not os.path.exists(sub_path) or not os.path.exists(num_path):
            continue

        try:
            sub = pd.read_csv(sub_path, sep='\t', low_memory=False)
            num = pd.read_csv(num_path, sep='\t', low_memory=False)
            sub_filtered = sub[sub['cik'] == cik]
            if sub_filtered.empty:
                continue

            adsh_list = sub_filtered['adsh'].unique()
            num_filtered = num[num['adsh'].isin(adsh_list)]

            output = pd.DataFrame()
            for label, tag_list in tags_needed.items():
                subset = num_filtered[num_filtered['tag'].isin(tag_list)]
                if not subset.empty:
                    summary = subset.groupby('ddate')['value'].sum().reset_index()
                    summary.columns = ['ddate', label]
                    if output.empty:
                        output = summary
                    else:
                        output = pd.merge(output, summary, on='ddate', how='outer')

            if not output.empty:
                output['ddate'] = pd.to_datetime(output['ddate'], format='%Y%m%d', errors='coerce')
                output['Quarter'] = quarter
                all_data.append(output)

        except Exception as e:
            print(f"⚠️ Error in {quarter} for {ticker}: {e}")

    if all_data:
        final_df = pd.concat(all_data).sort_values('ddate', ascending=False).reset_index(drop=True)
        output_dir = os.path.join(stock_root, ticker)
        os.makedirs(output_dir, exist_ok=True)
        output_file = os.path.join(output_dir, f"{ticker}_financial_summary.xlsx")
        final_df.to_excel(output_file, index=False)
        print(f"✅ {ticker} 저장 완료: {output_file}")
    else:
        print(f"❌ {ticker} 관련 데이터 없음.")


In [ ]:
import pandas as pd
import os

# 사용자 설정
base_folder = r"C:\Users\LabPC\OneDrive\주식/Financial Data"
tickers = {
    "AAPL": 320193,
}

tags_needed = {
    "Revenue": ["Revenues", "RevenueFromContractWithCustomerExcludingAssessedTax"],
    "NetIncome": ["NetIncomeLoss"],
    "TotalAssets": ["Assets"],
    "TotalEquity": ["StockholdersEquity", "StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest"],
    "TotalLiabilities": ["Liabilities"],
    "OperatingCashFlow": ["NetCashProvidedByUsedInOperatingActivities"],
    "CapEx": ["PaymentsToAcquirePropertyPlantAndEquipment"],
    "EBITDA": ["OperatingIncomeLoss"],
    "SharesOutstanding": ["WeightedAverageNumberOfDilutedSharesOutstanding"],
    "EPS": ["EarningsPerShareDiluted"],
    "TotalDebt": ["LongTermDebt", "Debt"],
}

quarters = sorted([f for f in os.listdir(base_folder) if f.startswith("20") and os.path.isdir(os.path.join(base_folder, f))])
stock_root = os.path.join(base_folder, "Stocks")
os.makedirs(stock_root, exist_ok=True)

for ticker, cik in tickers.items():
    all_data = []

    for quarter in quarters:
        quarter_path = os.path.join(base_folder, quarter)
        sub_path = os.path.join(quarter_path, "sub.txt")
        num_path = os.path.join(quarter_path, "num.txt")

        if not os.path.exists(sub_path) or not os.path.exists(num_path):
            continue

        try:
            sub = pd.read_csv(sub_path, sep='\t', low_memory=False)
            num = pd.read_csv(num_path, sep='\t', low_memory=False)
            sub = sub[sub['cik'] == cik]

            if sub.empty:
                continue

            # 🎯 전략 1: 최신 제출된 보고서만 유지 (form + period 기준)
            sub = sub.sort_values("filed", ascending=False)
            sub = sub.drop_duplicates(subset=["form", "period"])

            # adsh + form 매핑
            adsh_form_map = sub[["adsh", "form"]].drop_duplicates()
            num_filtered = num[num["adsh"].isin(adsh_form_map["adsh"])]
            num_merged = pd.merge(num_filtered, adsh_form_map, on="adsh", how="left")

            output = pd.DataFrame()
            for label, tag_list in tags_needed.items():
                subset = num_merged[num_merged["tag"].isin(tag_list)]
                if not subset.empty:
                    summary = subset.groupby(["ddate", "form"])["value"].sum().reset_index()
                    summary.columns = ["ddate", "FormType", label]
                    if output.empty:
                        output = summary
                    else:
                        output = pd.merge(output, summary, on=["ddate", "FormType"], how="outer")

            if not output.empty:
                output["ddate"] = pd.to_datetime(output["ddate"], format="%Y%m%d", errors="coerce")
                output["Quarter"] = quarter

                # 🔧 중복 제거 (form + ddate 기준)
                output = output.groupby(["ddate", "FormType", "Quarter"], as_index=False).first()
                all_data.append(output)

        except Exception as e:
            print(f"⚠️ Error in {quarter} for {ticker}: {e}")

    if all_data:
        final_df = pd.concat(all_data).sort_values("ddate", ascending=False).reset_index(drop=True)
        output_dir = os.path.join(stock_root, ticker)
        os.makedirs(output_dir, exist_ok=True)
        output_file = os.path.join(output_dir, f"{ticker}_financial_summary.xlsx")
        final_df.to_excel(output_file, index=False)
        print(f"✅ {ticker} 저장 완료: {output_file}")
    else:
        print(f"❌ {ticker} 관련 데이터 없음.")


In [ ]:
import pandas as pd
import os

# 사용자 설정
base_folder = r"C:\Users\LabPC\OneDrive\주식/Financial Data"
tickers = {
    "AAPL": 320193,
}

tags_needed = {
    "Revenue": ["Revenues", "RevenueFromContractWithCustomerExcludingAssessedTax"],
    "NetIncome": ["NetIncomeLoss"],
    "TotalAssets": ["Assets"],
    "TotalEquity": ["StockholdersEquity", "StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest"],
    "TotalLiabilities": ["Liabilities"],
    "OperatingCashFlow": ["NetCashProvidedByUsedInOperatingActivities"],
    "CapEx": ["PaymentsToAcquirePropertyPlantAndEquipment"],
    "EBITDA": ["OperatingIncomeLoss"],
    "SharesOutstanding": ["WeightedAverageNumberOfDilutedSharesOutstanding"],
    "EPS": ["EarningsPerShareDiluted"],
    "TotalDebt": ["LongTermDebt", "Debt"],
}

# 분기 폴더 목록(예: "2009q1", "2009q2", …)
quarters = sorted([f for f in os.listdir(base_folder)
                   if f.startswith("20") and os.path.isdir(os.path.join(base_folder, f))])

stock_root = os.path.join(base_folder, "Stocks")
os.makedirs(stock_root, exist_ok=True)

for ticker, cik in tickers.items():
    all_data = []

    for quarter in quarters:
        sub_path = os.path.join(base_folder, quarter, "sub.txt")
        num_path = os.path.join(base_folder, quarter, "num.txt")

        if not (os.path.exists(sub_path) and os.path.exists(num_path)):
            continue

        try:
            sub = pd.read_csv(sub_path, sep='\t', low_memory=False)
            num = pd.read_csv(num_path, sep='\t', low_memory=False)
            sub = sub[sub['cik'] == cik]
            if sub.empty:
                continue

            # 최신 제출만: form+period 기준으로 filed 내림차순 정렬 후 중복 제거
            sub = sub.sort_values("filed", ascending=False)
            sub = sub.drop_duplicates(subset=["form", "period"])

            # adsh ↔ form 매핑
            adsh_form = sub[['adsh','form']].drop_duplicates()
            num_f = num[num['adsh'].isin(adsh_form['adsh'])]
            merged = pd.merge(num_f, adsh_form, on='adsh', how='left')

            # 필요한 태그별 집계
            output = pd.DataFrame()
            for label, tags in tags_needed.items():
                tmp = merged[merged['tag'].isin(tags)]
                if not tmp.empty:
                    s = tmp.groupby(['ddate','form'])['value'].sum().reset_index()
                    s.columns = ['ddate','FormType', label]
                    output = s if output.empty else pd.merge(output, s, on=['ddate','FormType'], how='outer')

            if not output.empty:
                output['ddate'] = pd.to_datetime(output['ddate'], format='%Y%m%d', errors='coerce')
                all_data.append(output)

        except Exception as e:
            print(f"⚠️ Error in {quarter} for {ticker}: {e}")

    # 수집된 모든 분기 데이터 합치기
    if all_data:
        final_df = pd.concat(all_data) \
                     .sort_values('ddate', ascending=False) \
                     .reset_index(drop=True)

        # ➡️ 폴더(Quarter) 무시: ddate+FormType 기준으로 중복 제거
        final_df = final_df.groupby(['ddate','FormType'], as_index=False).first()

        # 결과 저장
        out_dir = os.path.join(stock_root, ticker)
        os.makedirs(out_dir, exist_ok=True)
        out_file = os.path.join(out_dir, f"{ticker}_financial_summary.xlsx")
        final_df.to_excel(out_file, index=False)
        print(f"✅ {ticker} 저장 완료: {out_file}")
    else:
        print(f"❌ {ticker} 관련 데이터 없음.")


In [ ]:
import pandas as pd
import os

# 사용자 설정
base_folder = r"C:\Users\LabPC\OneDrive\주식/Financial Data"
tickers = {
    "AAPL": 320193,
}

tags_needed = {
    "Revenue": ["Revenues", "RevenueFromContractWithCustomerExcludingAssessedTax"],
    "NetIncome": ["NetIncomeLoss"],
    "TotalAssets": ["Assets"],
    "TotalEquity": ["StockholdersEquity", "StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest"],
    "TotalLiabilities": ["Liabilities"],
    "OperatingCashFlow": ["NetCashProvidedByUsedInOperatingActivities"],
    "CapEx": ["PaymentsToAcquirePropertyPlantAndEquipment"],
    "EBITDA": ["OperatingIncomeLoss"],
    "SharesOutstanding": ["WeightedAverageNumberOfDilutedSharesOutstanding"],
    "EPS": ["EarningsPerShareDiluted"],
    "TotalDebt": ["LongTermDebt", "Debt"],
}

# 분기 폴더 목록(예: "2009q1", "2009q2", …)
quarters = sorted([
    f for f in os.listdir(base_folder)
    if f.startswith("20") and os.path.isdir(os.path.join(base_folder, f))
])

stock_root = os.path.join(base_folder, "Stocks")
os.makedirs(stock_root, exist_ok=True)

for ticker, cik in tickers.items():
    all_data = []

    for quarter in quarters:
        sub_path = os.path.join(base_folder, quarter, "sub.txt")
        num_path = os.path.join(base_folder, quarter, "num.txt")
        if not (os.path.exists(sub_path) and os.path.exists(num_path)):
            continue

        try:
            sub = pd.read_csv(sub_path, sep='\t', low_memory=False)
            num = pd.read_csv(num_path, sep='\t', low_memory=False)
            sub = sub[sub['cik'] == cik]
            if sub.empty:
                continue

            # 최신 제출만: form+period 기준으로 정렬 후 중복 제거
            sub = sub.sort_values("filed", ascending=False)
            sub = sub.drop_duplicates(subset=["form", "period"])

            # adsh ↔ form 매핑
            adsh_form = sub[['adsh', 'form']].drop_duplicates()
            num_f = num[num['adsh'].isin(adsh_form['adsh'])]
            merged = pd.merge(num_f, adsh_form, on='adsh', how='left')

            # 필요한 태그별 집계
            output = pd.DataFrame()
            for label, tags in tags_needed.items():
                tmp = merged[merged['tag'].isin(tags)]
                if not tmp.empty:
                    s = tmp.groupby(['ddate', 'form'])['value'].sum().reset_index()
                    s.columns = ['ddate', 'FormType', label]
                    output = s if output.empty else pd.merge(output, s, on=['ddate', 'FormType'], how='outer')

            if not output.empty:
                output['ddate'] = pd.to_datetime(output['ddate'], format='%Y%m%d', errors='coerce')
                all_data.append(output)

        except Exception as e:
            print(f"⚠️ Error in {quarter} for {ticker}: {e}")

    # 수집된 모든 분기 데이터 합치기
    if not all_data:
        print(f"❌ {ticker} 관련 데이터 없음.")
        continue

    final_df = pd.concat(all_data) \
                 .sort_values('ddate', ascending=False) \
                 .reset_index(drop=True)

    # ➡️ 폴더(Quarter) 무시: ddate+FormType 기준으로 중복 제거
    final_df = final_df.groupby(['ddate', 'FormType'], as_index=False).first()

    # 10-Q와 10-K를 분리
    df_10q = final_df[final_df['FormType'].str.startswith('10-Q')].copy()
    df_10k = final_df[final_df['FormType'].str.startswith('10-K')].copy()

    # 저장 디렉토리
    out_dir = os.path.join(stock_root, ticker)
    os.makedirs(out_dir, exist_ok=True)

    # 10-Q 파일로 저장
    if not df_10q.empty:
        q_file = os.path.join(out_dir, f"{ticker}_financial_Q.xlsx")
        df_10q.to_excel(q_file, index=False)
        print(f"✅ {ticker} 10-Q 저장 완료: {q_file}")
    else:
        print(f"⚠️ {ticker} 10-Q 데이터 없음")

    # 10-K 파일로 저장
    if not df_10k.empty:
        k_file = os.path.join(out_dir, f"{ticker}_financial_K.xlsx")
        df_10k.to_excel(k_file, index=False)
        print(f"✅ {ticker} 10-K 저장 완료: {k_file}")
    else:
        print(f"⚠️ {ticker} 10-K 데이터 없음")


In [ ]:
new tageed

In [ ]:
import os
import glob
import pandas as pd
import numpy as np

# ─────────────────────────────────────────────────────────────
# 사용자 설정
# ─────────────────────────────────────────────────────────────
base_folder    = r"C:\Users\LabPC\OneDrive\주식/Financial Data"
stock_root     = os.path.join(base_folder, "Stocks")
os.makedirs(stock_root, exist_ok=True)

tickers = {
    "AAPL": 320193,
}

# ─────────────────────────────────────────────────────────────
# 검색할 태그들 (더 많은 변형명을 포함하여 정확도 향상)
# ─────────────────────────────────────────────────────────────
tags_needed = {
    "Revenue": [
        "Revenues",
        "SalesRevenueNet",
        "RevenuesNetOfDutiesAndTaxes",
        "RevenueFromContractWithCustomerExcludingAssessedTax"
    ],
    "NetIncome": [
        "NetIncomeLoss",
        "ProfitLoss",
        "NetIncome"
    ],
    "TotalAssets": [
        "Assets",
        "AssetsCurrent",
        "AssetsNoncurrent"
    ],
    "TotalLiabilities": [
        "Liabilities",
        "LiabilitiesCurrent",
        "LiabilitiesNoncurrent"
    ],
    "TotalEquity": [
        "StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest",
        "StockholdersEquity"
    ],
    "OperatingCashFlow": [
        "NetCashProvidedByUsedInOperatingActivities",
        "CashProvidedByUsedInOperatingActivities"
    ],
    "CapEx": [
        "PaymentsToAcquirePropertyPlantAndEquipment",
        "AdditionsToPropertyPlantAndEquipment"
    ],
    # EBITDA = OperatingIncomeLoss + DepreciationAndAmortization
    "OperatingIncomeLoss": ["OperatingIncomeLoss"],
    "DepreciationAndAmortization": ["DepreciationAndAmortization"],
    "SharesOutstanding": [
        "WeightedAverageNumberOfDilutedSharesOutstanding",
        "CommonStockSharesOutstanding"
    ],
    "EPS": [
        "EarningsPerShareDiluted",
        "DilutedEarningsPerShare"
    ],
    "TotalDebt": [
        "LongTermDebtNoncurrent",
        "LongTermDebtAndCapitalLeaseObligations",
        "Debt"
    ],
}

# ─────────────────────────────────────────────────────────────
# 분기별 폴더 목록
# ─────────────────────────────────────────────────────────────
quarters = sorted([
    f for f in os.listdir(base_folder)
    if f.startswith("20") and os.path.isdir(os.path.join(base_folder, f))
])

for ticker, cik in tickers.items():
    all_data = []

    for quarter in quarters:
        sub_path = os.path.join(base_folder, quarter, "sub.txt")
        num_path = os.path.join(base_folder, quarter, "num.txt")
        if not (os.path.exists(sub_path) and os.path.exists(num_path)):
            continue

        sub = pd.read_csv(sub_path, sep="\t", low_memory=False)
        num = pd.read_csv(num_path, sep="\t", low_memory=False)

        sub = sub[sub["cik"] == cik]
        if sub.empty:
            continue

        # 최신 제출만: 중복된 form+period 제거
        sub = sub.sort_values("filed", ascending=False)
        sub = sub.drop_duplicates(subset=["form", "period"])

        # adsh ↔ form 매핑
        adsh_form = sub.loc[:, ["adsh", "form"]].drop_duplicates()
        num_f = num[num["adsh"].isin(adsh_form["adsh"])]
        merged = num_f.merge(adsh_form, on="adsh", how="left")

        # 각 태그별 집계
        out = pd.DataFrame()
        for label, tag_list in tags_needed.items():
            tmp = merged[merged["tag"].isin(tag_list)]
            if not tmp.empty:
                s = (
                    tmp
                    .groupby(["ddate", "form"], as_index=False)["value"]
                    .sum()
                )
                s = s.rename(columns={"value": label, "form": "FormType"})
                if out.empty:
                    out = s
                else:
                    out = out.merge(s, on=["ddate", "FormType"], how="outer")

        if not out.empty:
            out["ddate"] = pd.to_datetime(out["ddate"], format="%Y%m%d", errors="coerce")
            all_data.append(out)

    if not all_data:
        print(f"❌ {ticker} 관련 데이터 없음.")
        continue

    # 전체 합치고 최신 기준으로 중복 제거
    df_all = pd.concat(all_data, ignore_index=True)
    df_all = (
        df_all
        .sort_values("ddate", ascending=False)
        .drop_duplicates(subset=["ddate", "FormType"])
        .reset_index(drop=True)
    )

    # 10-Q / 10-K 분리
    df_10q = df_all[df_all["FormType"].str.startswith("10-Q")].copy()
    df_10k = df_all[df_all["FormType"].str.startswith("10-K")].copy()

    # EBITDA 계산: OperatingIncomeLoss + DepreciationAndAmortization
    for df in (df_10q, df_10k):
        if {"OperatingIncomeLoss", "DepreciationAndAmortization"}.issubset(df.columns):
            df["EBITDA"] = df["OperatingIncomeLoss"] + df["DepreciationAndAmortization"]
            # 원래 개별 컬럼은 유지하거나 삭제 가능
            df.drop(columns=["OperatingIncomeLoss", "DepreciationAndAmortization"], inplace=True)

    # 저장
    out_dir = os.path.join(stock_root, ticker)
    os.makedirs(out_dir, exist_ok=True)

    if not df_10q.empty:
        q_file = os.path.join(out_dir, f"{ticker}_financial_Q.xlsx")
        df_10q.to_excel(q_file, index=False)
        print(f"✅ {ticker} 10-Q 저장 완료: {q_file}")
    else:
        print(f"⚠️ {ticker} 10-Q 데이터 없음")

    if not df_10k.empty:
        k_file = os.path.join(out_dir, f"{ticker}_financial_K.xlsx")
        df_10k.to_excel(k_file, index=False)
        print(f"✅ {ticker} 10-K 저장 완료: {k_file}")
    else:
        print(f"⚠️ {ticker} 10-K 데이터 없음")


In [ ]:
import os
import glob
import pandas as pd
import numpy as np

# ─────────────────────────────────────────────────────────────
# 사용자 설정
# ─────────────────────────────────────────────────────────────
base_folder    = r"C:\Users\LabPC\OneDrive\주식/Financial Data"
stock_root     = os.path.join(base_folder, "Stocks")
os.makedirs(stock_root, exist_ok=True)

tickers = {
    "AAPL": 320193,
}

# ─────────────────────────────────────────────────────────────
# 검색할 태그들
# ─────────────────────────────────────────────────────────────
tags_needed = {
    "Revenue": [
        "Revenues", "SalesRevenueNet", "RevenuesNetOfDutiesAndTaxes", "RevenueFromContractWithCustomerExcludingAssessedTax"
    ],
    "NetIncome": [
        "NetIncomeLoss", "ProfitLoss", "NetIncome"
    ],
    "TotalAssets": [
        "Assets", "AssetsCurrent", "AssetsNoncurrent"
    ],
    "TotalLiabilities": [
        "Liabilities", "LiabilitiesCurrent", "LiabilitiesNoncurrent"
    ],
    "TotalEquity": [
        "StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest", "StockholdersEquity"
    ],
    "OperatingCashFlow": [
        "NetCashProvidedByUsedInOperatingActivities", "CashProvidedByUsedInOperatingActivities"
    ],
    "CapEx": [
        "PaymentsToAcquirePropertyPlantAndEquipment", "AdditionsToPropertyPlantAndEquipment"
    ],
    "OperatingIncomeLoss": ["OperatingIncomeLoss"],
    "DepreciationAndAmortization": ["DepreciationAndAmortization"],
    "SharesOutstanding": [
        "WeightedAverageNumberOfDilutedSharesOutstanding", "CommonStockSharesOutstanding"
    ],
    "EPS": [
        "EarningsPerShareDiluted", "DilutedEarningsPerShare"
    ],
    "TotalDebt": [
        "LongTermDebtNoncurrent", "LongTermDebtAndCapitalLeaseObligations", "Debt"
    ],
}

# ─────────────────────────────────────────────────────────────
# 분기별 폴더 목록
# ─────────────────────────────────────────────────────────────
quarters = sorted([
    f for f in os.listdir(base_folder)
    if f.startswith("20") and os.path.isdir(os.path.join(base_folder, f))
])

# ─────────────────────────────────────────────────────────────
# 메인 루프
# ─────────────────────────────────────────────────────────────
for ticker, cik in tickers.items():
    print(f"\n🔍 시작: {ticker} (CIK: {cik})")
    all_data = []

    for quarter in quarters:
        print(f"  📁 처리 중: {quarter}")
        sub_path = os.path.join(base_folder, quarter, "sub.txt")
        num_path = os.path.join(base_folder, quarter, "num.txt")

        if not (os.path.exists(sub_path) and os.path.exists(num_path)):
            print("    ⚠️ 파일 누락 - 건너뜀")
            continue

        print("    📄 파일 읽는 중...")
        sub = pd.read_csv(sub_path, sep="\t", low_memory=False)
        num = pd.read_csv(num_path, sep="\t", low_memory=False)

        sub = sub[sub["cik"] == cik]
        if sub.empty:
            print("    ⚠️ 해당 기업 데이터 없음 - 건너뜀")
            continue

        sub = sub.sort_values("filed", ascending=False)
        sub = sub.drop_duplicates(subset=["form", "period"])
        adsh_form = sub.loc[:, ["adsh", "form"]].drop_duplicates()
        num_f = num[num["adsh"].isin(adsh_form["adsh"])]
        merged = num_f.merge(adsh_form, on="adsh", how="left")

        print("    🧮 태그 집계 중...")
        out = pd.DataFrame()
        for label, tag_list in tags_needed.items():
            tmp = merged[merged["tag"].isin(tag_list)]
            if not tmp.empty:
                s = (
                    tmp
                    .groupby(["ddate", "form"], as_index=False)["value"]
                    .sum()
                )
                s = s.rename(columns={"value": label, "form": "FormType"})
                if out.empty:
                    out = s
                else:
                    out = out.merge(s, on=["ddate", "FormType"], how="outer")

        if not out.empty:
            out["ddate"] = pd.to_datetime(out["ddate"], format="%Y%m%d", errors="coerce")
            all_data.append(out)
            print("    ✅ 완료됨.")
        else:
            print("    ⚠️ 유효한 태그 없음 - 건너뜀")

    if not all_data:
        print(f"❌ {ticker} 관련 데이터 없음.")
        continue

    print("🔗 전체 데이터 병합 및 정리 중...")
    df_all = pd.concat(all_data, ignore_index=True)
    df_all = (
        df_all
        .sort_values("ddate", ascending=False)
        .drop_duplicates(subset=["ddate", "FormType"])
        .reset_index(drop=True)
    )

    df_10q = df_all[df_all["FormType"].str.startswith("10-Q")].copy()
    df_10k = df_all[df_all["FormType"].str.startswith("10-K")].copy()

    print("🧾 EBITDA 계산 중...")
    for df in (df_10q, df_10k):
        if {"OperatingIncomeLoss", "DepreciationAndAmortization"}.issubset(df.columns):
            df["EBITDA"] = df["OperatingIncomeLoss"] + df["DepreciationAndAmortization"]
            df.drop(columns=["OperatingIncomeLoss", "DepreciationAndAmortization"], inplace=True)

    out_dir = os.path.join(stock_root, ticker)
    os.makedirs(out_dir, exist_ok=True)

    if not df_10q.empty:
        q_file = os.path.join(out_dir, f"{ticker}_financial_Q.xlsx")
        df_10q.to_excel(q_file, index=False)
        print(f"💾 저장 완료: {q_file}")
    else:
        print(f"⚠️ {ticker} 10-Q 데이터 없음")

    if not df_10k.empty:
        k_file = os.path.join(out_dir, f"{ticker}_financial_K.xlsx")
        df_10k.to_excel(k_file, index=False)
        print(f"💾 저장 완료: {k_file}")
    else:
        print(f"⚠️ {ticker} 10-K 데이터 없음")


In [ ]:
## Test
C:\Users\seung\OneDrive\주식\Financial Data\test\2024q4

In [ ]:
import os
import pandas as pd

# 경로 설정 (이미 2024q4 지정되어 있음)
base_folder = r"C:\Users\LabPC\OneDrive\주식\Financial Data\test\2025q1"
cik = 320193
output_file = os.path.join(base_folder, f"AAPL_all_tags.xlsx")

# 파일 경로
num_path = os.path.join(base_folder, "num.txt")
sub_path = os.path.join(base_folder, "sub.txt")

# 파일 로드
num = pd.read_csv(num_path, sep="\t", low_memory=False)
sub = pd.read_csv(sub_path, sep="\t", low_memory=False)

# 애플 관련 adsh 추출
apple_adsh_list = sub[sub["cik"] == cik]["adsh"].unique()
apple_num = num[num["adsh"].isin(apple_adsh_list)]

# 필요한 열만 추출
cols_to_show = ["adsh", "tag", "version", "ddate", "qtrs", "uom", "value", "footnote"]
apple_data = apple_num[cols_to_show].copy()

# 날짜 정리
apple_data["ddate"] = pd.to_datetime(apple_data["ddate"], format="%Y%m%d", errors="coerce")

# 중복 제거 (tag + ddate 기준)
apple_data = apple_data.sort_values("ddate", ascending=False)
apple_data = apple_data.drop_duplicates(subset=["tag", "ddate"])

# 보기 쉽게 정렬
apple_data = apple_data.sort_values(["tag", "ddate"])

# 저장
apple_data.to_excel(output_file, index=False)
print(f"📊 전체 태그 저장 완료: {output_file}")


In [ ]:
import os
import pandas as pd
import numpy as np

# ────────────────────────────────────────────────
# 📁 사용자 설정
# ────────────────────────────────────────────────
base_folder = r"C:\Users\LabPC\OneDrive\주식\Financial Data\test"
stock_root = os.path.join(base_folder, "Stocks")
os.makedirs(stock_root, exist_ok=True)

tickers = {
    "AAPL": 320193,
}

main_revenue_tag = "RevenueFromContractWithCustomerExcludingAssessedTax"

# 분기 폴더 리스트 예: 2025q1, 2025q2, ...
quarters = sorted([
    f for f in os.listdir(base_folder)
    if f.startswith("20") and os.path.isdir(os.path.join(base_folder, f))
])

all_quarterly_revenue = []
all_revenue_segments = []

for ticker, cik in tickers.items():
    print(f"\n🔍 시작: {ticker} (CIK: {cik})")

    for quarter in quarters:
        folder_path = os.path.join(base_folder, quarter)
        sub_path = os.path.join(folder_path, "sub.txt")
        num_path = os.path.join(folder_path, "num.txt")

        if not (os.path.exists(sub_path) and os.path.exists(num_path)):
            print(f"⚠️ 파일 누락: {quarter}")
            continue

        try:
            sub = pd.read_csv(sub_path, sep="\t", low_memory=False)
            num = pd.read_csv(num_path, sep="\t", low_memory=False)
        except Exception as e:
            print(f"⚠️ 읽기 오류: {quarter}: {e}")
            continue

        sub = sub[sub["cik"] == cik]
        if sub.empty:
            continue

        adsh_form = sub[["adsh", "form", "period"]].drop_duplicates()
        num = num[num["adsh"].isin(adsh_form["adsh"])]
        merged = num.merge(adsh_form, on="adsh", how="left")

        # 전체 매출
        full_rev = merged[merged["tag"] == main_revenue_tag]
        full_rev = full_rev[["ddate", "qtrs", "value", "period", "form"]]
        full_rev["value"] = pd.to_numeric(full_rev["value"], errors="coerce")
        full_rev = full_rev.groupby(["ddate", "qtrs", "period", "form"]).agg({"value": "sum"}).reset_index()
        full_rev = full_rev.sort_values("ddate")

        # q2 - q1 분기 추출
        q2_data = full_rev[full_rev["qtrs"] == 2]
        q1_data = full_rev[full_rev["qtrs"] == 1]
        q_merged = pd.merge(
            q2_data, q1_data, on="period", suffixes=("_q2", "_q1")
        )
        if not q_merged.empty:
            q_merged["quarter_revenue"] = q_merged["value_q2"] - q_merged["value_q1"]
            q_merged["period"] = pd.to_datetime(q_merged["period"], format="%Y%m%d")
            all_quarterly_revenue.append(q_merged[["period", "quarter_revenue"]])

        # 세그먼트별 매출 수집 (segment 열이 있는 경우에만)
        if "segment" in merged.columns:
            detail_rev = merged[merged["tag"] == main_revenue_tag]
            detail_rev = detail_rev[["ddate", "qtrs", "segment", "value", "period"]]
            detail_rev["value"] = pd.to_numeric(detail_rev["value"], errors="coerce")
            detail_rev = detail_rev[~detail_rev["segment"].isna()]
            detail_rev["period"] = pd.to_datetime(detail_rev["period"], format="%Y%m%d")
            all_revenue_segments.append(detail_rev)
        else:
            print(f"⚠️ {quarter}에는 'segment' 열이 없어 세그먼트별 분석 생략")

# 데이터 병합 및 출력
if all_quarterly_revenue:
    df_quarterly = pd.concat(all_quarterly_revenue).sort_values("period").reset_index(drop=True)
    print("\n📈 분기별 Revenue 계산 결과:")
    display(df_quarterly)
else:
    print("\n⚠️ qtrs=1,2 조건을 만족하는 분기 데이터가 없어 분기 revenue 계산 불가")

if all_revenue_segments:
    df_segments = pd.concat(all_revenue_segments).reset_index(drop=True)
    df_segments_q1 = df_segments[df_segments["qtrs"] == 1]
    df_segments_pivot = df_segments_q1.pivot_table(
        index="period", columns="segment", values="value", aggfunc="sum"
    ).reset_index()
    print("\n📊 세그먼트별 Revenue 분기 분석 (qtrs=1 기준):")
    display(df_segments_pivot)
else:
    print("\n⚠️ 세그먼트 데이터가 없습니다.")


In [ ]:
# 8/7C:\Users\LabPC\OneDrive\주식\Financial Data\2025q2

# Revenue finding (Quarterly)

In [ ]:
import os
import pandas as pd
import numpy as np

# ─────────────────────────────────────────────
# 🔧 사용자 설정
# ─────────────────────────────────────────────
base_folder = r"C:\Users\LabPC\OneDrive\주식\Financial Data\test"
main_tag = "RevenueFromContractWithCustomerExcludingAssessedTax"
cik_dict = {
    "Apple": "0000320193",
    "Tesla": "0001318605"
}

# 분기별 폴더 리스트 찾기
quarters = sorted([
    f for f in os.listdir(base_folder)
    if f.startswith("20") and os.path.isdir(os.path.join(base_folder, f))
])

# 회사별 루프
for company, cik_prefix in cik_dict.items():
    print(f"\n🔍 회사: {company} (CIK: {cik_prefix})")
    all_data = []

    for quarter in quarters:
        num_path = os.path.join(base_folder, quarter, "num.txt")
        if not os.path.exists(num_path):
            print(f"❌ 누락: {quarter}")
            continue

        try:
            df = pd.read_csv(num_path, sep="\t", low_memory=False)
        except Exception as e:
            print(f"❌ 읽기 실패: {quarter} - {e}")
            continue

        # 매출 관련 태그 필터
        df = df[df["tag"] == main_tag].copy()

        # CIK 필터링
        df = df[df["adsh"].str.startswith(cik_prefix)]

        # 값 전처리
        df["value"] = pd.to_numeric(df["value"], errors="coerce")
        df["period"] = pd.to_datetime(df["ddate"], format="%Y%m%d", errors="coerce")
        df["segment"] = df["segments"].fillna("").str.extract(r"(ProductOrService|BusinessSegments)=(.*?);")[1]
        df["type"] = df["segments"].fillna("").str.extract(r"(ProductOrService|BusinessSegments)")[0]

        # 📎 source 컬럼 추가
        df["source"] = os.path.join(quarter, "num.txt")

        all_data.append(df)

    # ─────────────────────────────────────────────
    # 데이터 통합
    # ─────────────────────────────────────────────
    if not all_data:
        print(f"⚠️ {company}: 유효한 데이터 없음")
        continue

    df_all = pd.concat(all_data, ignore_index=True)

    # ⛔ qtrs == 1 데이터만 필터링
    df_all = df_all[df_all["qtrs"] == 1]

    # ─────────────────────────────────────────────
    # 전체 Revenue (segment 없음, 전체 합산용)
    # ─────────────────────────────────────────────
    df_total = df_all[df_all["segment"].isna()].copy()
    df_total = df_total.groupby(["period", "qtrs", "source"]).agg({"value": "sum"}).reset_index()
    df_total = df_total.sort_values("period")

    print(f"\n💰 {company} 전체 매출 (qtrs == 1):")
    display(df_total)

    # ─────────────────────────────────────────────
    # 📊 Pivot: ProductOrService
    # ─────────────────────────────────────────────
    df_prod = df_all[df_all["type"] == "ProductOrService"]
    pivot_prod = df_prod.pivot_table(
        index=["period", "qtrs", "source"], columns="segment", values="value", aggfunc="sum"
    ).reset_index()

    print(f"\n🛍 {company} ProductOrService 세그먼트 (qtrs == 1):")
    display(pivot_prod)

    # ─────────────────────────────────────────────
    # 📊 Pivot: BusinessSegments
    # ─────────────────────────────────────────────
    df_biz = df_all[df_all["type"] == "BusinessSegments"]
    pivot_biz = df_biz.pivot_table(
        index=["period", "qtrs", "source"], columns="segment", values="value", aggfunc="sum"
    ).reset_index()

    print(f"\n🌍 {company} BusinessSegments 세그먼트 (qtrs == 1):")
    display(pivot_biz)


In [ ]:
# 모든거 다 Finding

In [ ]:
import os
import pandas as pd
import numpy as np

# ─────────────────────────────────────────────
# 🔧 사용자 설정
# ─────────────────────────────────────────────
base_folder = r"C:\Users\LabPC\OneDrive\주식\Financial Data\test"
cik_dict = {
    "Apple": "0000320193"
}

# 항목별 tag 설정
main_tags = {
    "Revenue": ["Revenues", "SalesRevenueNet", "RevenuesNetOfDutiesAndTaxes", "RevenueFromContractWithCustomerExcludingAssessedTax"],
    "EPS": ["EarningsPerShareDiluted", "EarningsPerShareBasic"],
    "NetIncome": ["NetIncomeLoss", "ProfitLoss", "NetIncome"],
    "TotalAssets": ["Assets"],
    "TotalLiabilities": ["Liabilities"],
    "OperatingCashFlow": ["NetCashProvidedByUsedInOperatingActivities"],
    "CurrentAssets": ["AssetsCurrent"],
    "CurrentLiabilities": ["LiabilitiesCurrent"],
    "GrossProfit": ["GrossProfit"],
    "CapEx": ["PaymentsToAcquirePropertyPlantAndEquipment", "CapitalExpendituresIncurredButNotYetPaid"],
    "OperatingIncomeLoss": ["OperatingIncomeLoss"],
    "Depreciation": ["Depreciation"],
    "Amortization": ["Amortization"],
    # Equity 구성요소
    "CommonStockIncludingAdditionalPaidInCapital": ["CommonStockIncludingAdditionalPaidInCapital"],
    "RetainedEarnings": ["RetainedEarnings"],
    "AccumulatedOtherComprehensiveIncome": ["AccumulatedOtherComprehensiveIncome"]
}

# 분기별 폴더 리스트 찾기
quarters = sorted([
    f for f in os.listdir(base_folder)
    if f.startswith("20") and os.path.isdir(os.path.join(base_folder, f))
])

from IPython.display import display

def get_data(df, tags, cik_prefix, qtrs=None):
    df_tag = df[df["tag"].isin(tags)]
    df_tag = df_tag[df_tag["adsh"].str.startswith(cik_prefix)]
    if qtrs is not None:
        df_tag = df_tag[df_tag["qtrs"] == qtrs]
    return df_tag

for company, cik_prefix in cik_dict.items():
    print(f"\n🔍 회사: {company} (CIK: {cik_prefix})")
    all_quarter_data = []

    # 폴더별 데이터 통합
    for quarter in quarters:
        num_path = os.path.join(base_folder, quarter, "num.txt")
        if not os.path.exists(num_path):
            continue
        try:
            df = pd.read_csv(num_path, sep="\t", low_memory=False)
        except Exception:
            continue
        df["value"] = pd.to_numeric(df["value"], errors="coerce")
        df["qtrs"] = pd.to_numeric(df["qtrs"], errors="coerce")
        df["period"] = pd.to_datetime(df["ddate"], format="%Y%m%d", errors="coerce")
        df["source"] = os.path.join(quarter, "num.txt")
        # Revenue만 segment 정보 추출
        if any(tag in df["tag"].values for tag in main_tags["Revenue"]):
            df["segment"] = df["segments"].fillna("").str.extract(r"(ProductOrService|BusinessSegments)=(.*?);")[1]
            df["type"] = df["segments"].fillna("").str.extract(r"(ProductOrService|BusinessSegments)")[0]
        all_quarter_data.append(df)

    if not all_quarter_data:
        print(f"⚠️ {company}: 유효한 데이터 없음")
        continue

    df_all = pd.concat(all_quarter_data, ignore_index=True)

    # 1. Revenue 상세 breakdown
    revenue_df = get_data(df_all, main_tags["Revenue"], cik_prefix, qtrs=1)
    if not revenue_df.empty:
        print("\n💰 Revenue (qtrs==1, 전체 매출)")
        display(revenue_df[["period", "value", "tag", "segment", "type", "source"]])

        # (1) 전체 Revenue (segment 없음, 전체 합산용)
        df_total = revenue_df[revenue_df["segment"].isna()].copy()
        df_total = df_total.groupby(["period", "qtrs", "source"]).agg({"value": "sum"}).reset_index()
        df_total = df_total.sort_values("period")
        print("\n🧮 전체 Revenue (segment 없는 값 합산):")
        display(df_total)

        # (2) ProductOrService별
        df_prod = revenue_df[revenue_df["type"] == "ProductOrService"]
        pivot_prod = df_prod.pivot_table(
            index=["period", "qtrs", "source"], columns="segment", values="value", aggfunc="sum"
        ).reset_index()
        print("\n🛍 ProductOrService 세그먼트별 매출:")
        display(pivot_prod)

        # (3) BusinessSegments별
        df_biz = revenue_df[revenue_df["type"] == "BusinessSegments"]
        pivot_biz = df_biz.pivot_table(
            index=["period", "qtrs", "source"], columns="segment", values="value", aggfunc="sum"
        ).reset_index()
        print("\n🌍 BusinessSegments 세그먼트별 매출:")
        display(pivot_biz)

    # 2. EPS (qtrs==1)
    eps_df = get_data(df_all, main_tags["EPS"], cik_prefix, qtrs=1)
    print("\n📈 EPS (qtrs==1)")
    display(eps_df[["period", "value", "tag", "source"]])

    # 3. NetIncome (qtrs==1)
    netincome_df = get_data(df_all, main_tags["NetIncome"], cik_prefix, qtrs=1)
    print("\n🏦 NetIncome (qtrs==1)")
    display(netincome_df[["period", "value", "tag", "source"]])

    # 4. EstimatedEquity (qtrs==0, 3개 합산)
    equity_dfs = []
    for eq_tag in ["CommonStockIncludingAdditionalPaidInCapital", "RetainedEarnings", "AccumulatedOtherComprehensiveIncome"]:
        part = get_data(df_all, main_tags[eq_tag], cik_prefix, qtrs=0)
        equity_dfs.append(part.set_index("period")[["value"]].rename(columns={"value": eq_tag}))
    if equity_dfs:
        estimated_equity = pd.concat(equity_dfs, axis=1)
        estimated_equity["EstimatedEquity"] = estimated_equity.sum(axis=1)
        estimated_equity = estimated_equity.reset_index()
        print("\n🪙 EstimatedEquity (qtrs==0) [합산]")
        display(estimated_equity[["period", "CommonStockIncludingAdditionalPaidInCapital", "RetainedEarnings", "AccumulatedOtherComprehensiveIncome", "EstimatedEquity"]])

    # 5. TotalAssets (qtrs==0)
    ta_df = get_data(df_all, main_tags["TotalAssets"], cik_prefix, qtrs=0)
    print("\n🏢 TotalAssets (qtrs==0)")
    display(ta_df[["period", "value", "tag", "source"]])

    # 6. TotalLiabilities (qtrs==0)
    tl_df = get_data(df_all, main_tags["TotalLiabilities"], cik_prefix, qtrs=0)
    print("\n🏦 TotalLiabilities (qtrs==0)")
    display(tl_df[["period", "value", "tag", "source"]])

    # 7. OperatingCashFlow (qtrs==1)
    ocf_df = get_data(df_all, main_tags["OperatingCashFlow"], cik_prefix, qtrs=1)
    print("\n💵 OperatingCashFlow (qtrs==1)")
    display(ocf_df[["period", "value", "tag", "source"]])

    # 8. CurrentAssets (qtrs==0)
    ca_df = get_data(df_all, main_tags["CurrentAssets"], cik_prefix, qtrs=0)
    print("\n🗂 CurrentAssets (qtrs==0)")
    display(ca_df[["period", "value", "tag", "source"]])

    # 9. CurrentLiabilities (qtrs==0)
    cl_df = get_data(df_all, main_tags["CurrentLiabilities"], cik_prefix, qtrs=0)
    print("\n🗃 CurrentLiabilities (qtrs==0)")
    display(cl_df[["period", "value", "tag", "source"]])

    # 10. GrossProfit (qtrs==1)
    gp_df = get_data(df_all, main_tags["GrossProfit"], cik_prefix, qtrs=1)
    print("\n💵 GrossProfit (qtrs==1)")
    display(gp_df[["period", "value", "tag", "source"]])

    # 11. CapEx (qtrs==1)
    capex_df = get_data(df_all, main_tags["CapEx"], cik_prefix, qtrs=1)
    print("\n🏗 CapEx (qtrs==1)")
    display(capex_df[["period", "value", "tag", "source"]])

    # 12. OperatingIncomeLoss (qtrs==1)
    oil_df = get_data(df_all, main_tags["OperatingIncomeLoss"], cik_prefix, qtrs=1)
    print("\n🧾 OperatingIncomeLoss (qtrs==1)")
    display(oil_df[["period", "value", "tag", "source"]])

    # 13. Depreciation (qtrs==1)
    dep_df = get_data(df_all, main_tags["Depreciation"], cik_prefix, qtrs=1)
    print("\n🪙 Depreciation (qtrs==1)")
    display(dep_df[["period", "value", "tag", "source"]])

    # 14. Amortization (qtrs==1)
    amo_df = get_data(df_all, main_tags["Amortization"], cik_prefix, qtrs=1)
    print("\n🪙 Amortization (qtrs==1)")
    display(amo_df[["period", "value", "tag", "source"]])


In [ ]:
import os
import pandas as pd
import numpy as np

# ─────────────────────────────────────────────
# 🔧 사용자 설정
# ─────────────────────────────────────────────
base_folder = r"C:\Users\LabPC\OneDrive\주식\Financial Data\test"
cik_dict = {
    "Apple": "0000320193"
}

main_tags = {
    "Revenue": ["Revenues", "SalesRevenueNet", "RevenuesNetOfDutiesAndTaxes", "RevenueFromContractWithCustomerExcludingAssessedTax"],
    "EPS": ["EarningsPerShareDiluted", "EarningsPerShareBasic"],
    "NetIncome": ["NetIncomeLoss", "ProfitLoss", "NetIncome"],
    "TotalAssets": ["Assets"],
    "TotalLiabilities": ["Liabilities"],
    "OperatingCashFlow": ["NetCashProvidedByUsedInOperatingActivities"],
    "CurrentAssets": ["AssetsCurrent"],
    "CurrentLiabilities": ["LiabilitiesCurrent"],
    "GrossProfit": ["GrossProfit"],
    "CapEx": ["PaymentsToAcquirePropertyPlantAndEquipment"],  # CapEx는 이 태그만!
    "OperatingIncomeLoss": ["OperatingIncomeLoss"],           # 이 태그, qtrs=2
    "DepreciationAmortization": ["DepreciationDepletionAndAmortization"], # qtrs=2
    # Equity 구성요소
    "CommonStockIncludingAdditionalPaidInCapital": ["CommonStockIncludingAdditionalPaidInCapital"],
    "RetainedEarnings": ["RetainedEarnings"],
    "AccumulatedOtherComprehensiveIncome": ["AccumulatedOtherComprehensiveIncome"]
}

quarters = sorted([
    f for f in os.listdir(base_folder)
    if f.startswith("20") and os.path.isdir(os.path.join(base_folder, f))
])

from IPython.display import display

def get_data(df, tags, cik_prefix, qtrs=None):
    df_tag = df[df["tag"].isin(tags)]
    df_tag = df_tag[df_tag["adsh"].str.startswith(cik_prefix)]
    if qtrs is not None:
        df_tag = df_tag[df_tag["qtrs"] == qtrs]
    return df_tag

for company, cik_prefix in cik_dict.items():
    print(f"\n🔍 회사: {company} (CIK: {cik_prefix})")
    all_quarter_data = []

    for quarter in quarters:
        num_path = os.path.join(base_folder, quarter, "num.txt")
        if not os.path.exists(num_path):
            continue
        try:
            df = pd.read_csv(num_path, sep="\t", low_memory=False)
        except Exception:
            continue
        df["value"] = pd.to_numeric(df["value"], errors="coerce")
        df["qtrs"] = pd.to_numeric(df["qtrs"], errors="coerce")
        df["period"] = pd.to_datetime(df["ddate"], format="%Y%m%d", errors="coerce")
        df["source"] = os.path.join(quarter, "num.txt")
        if any(tag in df["tag"].values for tag in main_tags["Revenue"]):
            df["segment"] = df["segments"].fillna("").str.extract(r"(ProductOrService|BusinessSegments)=(.*?);")[1]
            df["type"] = df["segments"].fillna("").str.extract(r"(ProductOrService|BusinessSegments)")[0]
        all_quarter_data.append(df)

    if not all_quarter_data:
        print(f"⚠️ {company}: 유효한 데이터 없음")
        continue

    df_all = pd.concat(all_quarter_data, ignore_index=True)

    # 1. Revenue (qtrs==1)
    revenue_df = get_data(df_all, main_tags["Revenue"], cik_prefix, qtrs=1)
    if not revenue_df.empty:
        print("\n💰 Revenue (qtrs==1, 전체 매출)")
        display(revenue_df[["period", "value", "tag", "segment", "type", "source"]])

        df_total = revenue_df[revenue_df["segment"].isna()].copy()
        df_total = df_total.groupby(["period", "qtrs", "source"]).agg({"value": "sum"}).reset_index()
        df_total = df_total.sort_values("period")
        print("\n🧮 전체 Revenue (segment 없는 값 합산):")
        display(df_total)

        df_prod = revenue_df[revenue_df["type"] == "ProductOrService"]
        pivot_prod = df_prod.pivot_table(
            index=["period", "qtrs", "source"], columns="segment", values="value", aggfunc="sum"
        ).reset_index()
        print("\n🛍 ProductOrService 세그먼트별 매출:")
        display(pivot_prod)

        df_biz = revenue_df[revenue_df["type"] == "BusinessSegments"]
        pivot_biz = df_biz.pivot_table(
            index=["period", "qtrs", "source"], columns="segment", values="value", aggfunc="sum"
        ).reset_index()
        print("\n🌍 BusinessSegments 세그먼트별 매출:")
        display(pivot_biz)

    # 2. EPS (qtrs==1)
    eps_df = get_data(df_all, main_tags["EPS"], cik_prefix, qtrs=1)
    print("\n📈 EPS (qtrs==1)")
    display(eps_df[["period", "value", "tag", "source"]])

    # 3. NetIncome (qtrs==1)
    netincome_df = get_data(df_all, main_tags["NetIncome"], cik_prefix, qtrs=1)
    print("\n🏦 NetIncome (qtrs==1)")
    display(netincome_df[["period", "value", "tag", "source"]])

    # 4. EstimatedEquity (qtrs==0, 3개 합산)
    equity_dfs = []
    for eq_tag in ["CommonStockIncludingAdditionalPaidInCapital", "RetainedEarnings", "AccumulatedOtherComprehensiveIncome"]:
        part = get_data(df_all, main_tags[eq_tag], cik_prefix, qtrs=0)
        equity_dfs.append(part.set_index("period")[["value"]].rename(columns={"value": eq_tag}))
    if equity_dfs:
        estimated_equity = pd.concat(equity_dfs, axis=1)
        estimated_equity["EstimatedEquity"] = estimated_equity.sum(axis=1)
        estimated_equity = estimated_equity.reset_index()
        print("\n🪙 EstimatedEquity (qtrs==0) [합산]")
        display(estimated_equity[["period", "CommonStockIncludingAdditionalPaidInCapital", "RetainedEarnings", "AccumulatedOtherComprehensiveIncome", "EstimatedEquity"]])

    # 5. TotalAssets (qtrs==0)
    ta_df = get_data(df_all, main_tags["TotalAssets"], cik_prefix, qtrs=0)
    print("\n🏢 TotalAssets (qtrs==0)")
    display(ta_df[["period", "value", "tag", "source"]])

    # 6. TotalLiabilities (qtrs==0)
    tl_df = get_data(df_all, main_tags["TotalLiabilities"], cik_prefix, qtrs=0)
    print("\n🏦 TotalLiabilities (qtrs==0)")
    display(tl_df[["period", "value", "tag", "source"]])

    # 7. OperatingCashFlow (qtrs==1)
    ocf_df = get_data(df_all, main_tags["OperatingCashFlow"], cik_prefix, qtrs=1)
    print("\n💵 OperatingCashFlow (qtrs==1)")
    display(ocf_df[["period", "value", "tag", "source"]])

    # 8. CurrentAssets (qtrs==0)
    ca_df = get_data(df_all, main_tags["CurrentAssets"], cik_prefix, qtrs=0)
    print("\n🗂 CurrentAssets (qtrs==0)")
    display(ca_df[["period", "value", "tag", "source"]])

    # 9. CurrentLiabilities (qtrs==0)
    cl_df = get_data(df_all, main_tags["CurrentLiabilities"], cik_prefix, qtrs=0)
    print("\n🗃 CurrentLiabilities (qtrs==0)")
    display(cl_df[["period", "value", "tag", "source"]])

    # 10. GrossProfit (qtrs==1)
    gp_df = get_data(df_all, main_tags["GrossProfit"], cik_prefix, qtrs=1)
    print("\n💵 GrossProfit (qtrs==1)")
    display(gp_df[["period", "value", "tag", "source"]])

    # 11. CapEx (qtrs==1, PaymentsToAcquirePropertyPlantAndEquipment만)
    capex_df = get_data(df_all, main_tags["CapEx"], cik_prefix, qtrs=2)
    print("\n🏗 CapEx (qtrs==2, PaymentsToAcquirePropertyPlantAndEquipment)")
    display(capex_df[["period", "value", "tag", "source"]])

    # 12. OperatingIncomeLoss (qtrs==2)
    oil_df = get_data(df_all, main_tags["OperatingIncomeLoss"], cik_prefix, qtrs=2)
    print("\n🧾 OperatingIncomeLoss (qtrs==2)")
    display(oil_df[["period", "value", "tag", "source"]])

    # 13. Depreciation + Amortization (qtrs==2, DepreciationDepletionAndAmortization)
    depamo_df = get_data(df_all, main_tags["DepreciationAmortization"], cik_prefix, qtrs=2)
    print("\n🪙 Depreciation & Amortization (qtrs==2, DepreciationDepletionAndAmortization)")
    display(depamo_df[["period", "value", "tag", "source"]])


In [ ]:
import os
import pandas as pd
import numpy as np

# ─────────────────────────────────────────────
# 🔧 설정
# ─────────────────────────────────────────────
base_folder = r"C:\Users\LabPC\OneDrive\주식\Financial Data"
cik_dict = {
    "Apple": "0000320193",
    # "Tesla": "0001318605",
}

# 태그 후보 (가능한 alias 추가)
TAGS = {
    "Revenue": [
        "RevenueFromContractWithCustomerExcludingAssessedTax",
        "SalesRevenueNet", "Revenues", "RevenuesNetOfDutiesAndTaxes",
    ],
    "EPS": ["EarningsPerShareDiluted", "EarningsPerShareBasic"],  # qtrs=1
    "NetIncome": ["NetIncomeLoss", "ProfitLoss", "NetIncome"],    # qtrs=1

    "TotalAssets": ["Assets"],                  # qtrs=0
    "TotalLiabilities": ["Liabilities"],        # qtrs=0
    "CurrentAssets": ["AssetsCurrent"],         # qtrs=0
    "CurrentLiabilities": ["LiabilitiesCurrent"],# qtrs=0

    # Operating CF: 종종 ContinuingOperations 로만 찍힐 때가 있음
    "OperatingCashFlow": [
        "NetCashProvidedByUsedInOperatingActivities",
        "NetCashProvidedByUsedInOperatingActivitiesContinuingOperations",
    ],  # 보수적으로 qtrs 1,2,4 다 시도

    # GrossProfit 없으면 COGS 로 대체 계산
    "GrossProfit": ["GrossProfit"],             # qtrs=1
    "COGS": ["CostOfGoodsAndServicesSold", "CostOfRevenue"],

    # CapEx: 사용자 지정대로 이 태그만, qtrs는 2 우선 → 1 → 4 순으로 시도
    "CapEx": ["PaymentsToAcquirePropertyPlantAndEquipment"],

    # OI: 보수적으로 qtrs 2 우선 → 1
    "OperatingIncomeLoss": ["OperatingIncomeLoss"],

    # D&A: 통합 태그 우선, 없으면 보수적으로 몇 개 더 시도
    "DepreciationAmortization": [
        "DepreciationDepletionAndAmortization",
        "DepreciationAndAmortization",
        "DepreciationAmortizationAndAccretionNet",
    ],

    # Estimated Equity 구성요소 (qtrs=0)
    "APIC": ["CommonStockIncludingAdditionalPaidInCapital"],
    "RE": ["RetainedEarnings"],
    "AOCI": ["AccumulatedOtherComprehensiveIncome"],
}

# qtrs 허용값(우선순위 순서)
QPRS = {
    "Revenue": [1],
    "EPS": [1],
    "NetIncome": [1],

    "TotalAssets": [0],
    "TotalLiabilities": [0],
    "CurrentAssets": [0],
    "CurrentLiabilities": [0],

    "OperatingCashFlow": [1, 2, 4],
    "GrossProfit": [1],
    "COGS": [1],
    "CapEx": [2, 1, 4],
    "OperatingIncomeLoss": [2, 1],
    "DepreciationAmortization": [2, 1],
    "APIC": [0],
    "RE": [0],
    "AOCI": [0],
}

# ─────────────────────────────────────────────
# 🔧 헬퍼
# ─────────────────────────────────────────────
def load_all_num(base_folder):
    quarters = sorted([f for f in os.listdir(base_folder)
                       if f.startswith("20") and os.path.isdir(os.path.join(base_folder, f))])
    dfs = []
    for q in quarters:
        p = os.path.join(base_folder, q, "num.txt")
        if not os.path.exists(p):
            continue
        try:
            df = pd.read_csv(p, sep="\t", low_memory=False)
        except Exception:
            continue
        df["value"] = pd.to_numeric(df["value"], errors="coerce")
        df["qtrs"] = pd.to_numeric(df["qtrs"], errors="coerce")
        df["period"] = pd.to_datetime(df["ddate"], format="%Y%m%d", errors="coerce")
        df["source"] = os.path.join(q, "num.txt")
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

def filt(df, cik_prefix, tag_list, q_allowed, need_segment_null=True):
    if df.empty:
        return df
    out = df[df["tag"].isin(tag_list) & df["adsh"].str.startswith(cik_prefix)].copy()
    if q_allowed is not None:
        out = out[out["qtrs"].isin(q_allowed)]
    if need_segment_null:
        # 세그먼트가 없거나 완전 NaN 인 것만 사용 (사업부/제품 breakdown 제거)
        out = out[out["segments"].isna()]
    out = out.dropna(subset=["period"])
    out["Period"] = out["period"].dt.to_period("Q").astype(str)
    return out

def pick_first_by_period(df):
    # 같은 분기에 중복이 있으면 가장 첫 값만 사용
    return df.sort_values(["Period"]).groupby("Period", as_index=False).first()[["Period", "value"]]

def series_by_period(df):
    if df.empty:
        return pd.Series(dtype="float64")
    s = df.set_index("Period")["value"]
    # 같은 Period 중복이면 첫 값만
    s = s[~s.index.duplicated(keep="first")]
    return s

# ─────────────────────────────────────────────
# 📦 메인
# ─────────────────────────────────────────────
all_company_rows = []

num_all = load_all_num(base_folder)
if num_all.empty:
    raise SystemExit("num.txt 데이터를 못 찾았어요.")

for company, cik in cik_dict.items():
    df_all = num_all[num_all["adsh"].str.startswith(cik)].copy()

    # 1) Revenue: segment 없는 값만 합산 (동일 분기에 다수 보고 시 합계)
    rev = filt(df_all, cik, TAGS["Revenue"], QPRS["Revenue"], need_segment_null=True)
    rev = rev[rev["period"] >= pd.to_datetime("2009-01-01")]
    rev_sum = rev.groupby("Period", as_index=False)["value"].sum().rename(columns={"value": "Revenue"})
    periods = rev_sum["Period"].tolist()  # 기준 분기 (YYYYQn 문자열)

    # 2) 나머지 항목들: 태그/쿼터 후보를 우선순위대로 탐색해서 첫 값 사용
    def grab_metric(name, need_segment_null=True):
        cand_tags = TAGS[name]
        qcand = QPRS[name]
        got = pd.DataFrame(columns=["Period", "value"])
        for qv in qcand:
            if not got.empty:
                break
            df = filt(df_all, cik, cand_tags, [qv], need_segment_null=need_segment_null)
            if name == "GrossProfit" and df.empty:
                # GP 없으면 COGS로 대체 계산: GP = Revenue - COGS
                cogs = filt(df_all, cik, TAGS["COGS"], QPRS["COGS"], need_segment_null=True)
                if not cogs.empty:
                    cogs_s = series_by_period(cogs)
                    rev_s = rev_sum.set_index("Period")["Revenue"]
                    gp_calc = (rev_s - cogs_s).dropna()
                    if not gp_calc.empty:
                        got = gp_calc.reset_index().rename(columns={"index": "Period", 0: "value"})
                        break
            if not df.empty:
                got = pick_first_by_period(df)
        # Period 인덱스로 시리즈화
        s = series_by_period(got)
        # Period 기준으로 align
        out = pd.Series(index=periods, dtype="float64")
        out.loc[s.index.intersection(out.index)] = s.loc[s.index.intersection(out.index)]
        return out.values

    eps = grab_metric("EPS")
    ni = grab_metric("NetIncome")
    ta = grab_metric("TotalAssets")
    tl = grab_metric("TotalLiabilities")
    ca = grab_metric("CurrentAssets")
    cl = grab_metric("CurrentLiabilities")
    ocf = grab_metric("OperatingCashFlow")
    gp  = grab_metric("GrossProfit")
    capex = grab_metric("CapEx")
    oil = grab_metric("OperatingIncomeLoss")
    dda = grab_metric("DepreciationAmortization")

    # Estimated Equity = APIC + RE + AOCI (qtrs=0)
    apic = grab_metric("APIC")
    re_  = grab_metric("RE")
    aoci = grab_metric("AOCI")
    est_eq = np.nansum(np.vstack([apic, re_, aoci]), axis=0)

    # 회사/분기 단위 행 만들기
    df_company = pd.DataFrame({
        "Company": company,
        "Period": periods,
        "Revenue": rev_sum["Revenue"].values,
        "EPS": eps,
        "NetIncome": ni,
        "EstimatedEquity": est_eq,
        "TotalAssets": ta,
        "TotalLiabilities": tl,
        "OperatingCashFlow": ocf,
        "CurrentAssets": ca,
        "CurrentLiabilities": cl,
        "GrossProfit": gp,
        "CapEx": capex,
        "OperatingIncomeLoss": oil,
        "DepreciationAmortization": dda,
    })
    all_company_rows.append(df_company)

result_df = pd.concat(all_company_rows, ignore_index=True)

# 엑셀 저장
output_path = os.path.join(base_folder, "output.xlsx")
result_df.to_excel(output_path, index=False)
print(f"✅ 결과 저장: {output_path}")

# 확인
print(result_df.head(12))


In [ ]:
# Test

In [ ]:
# import os
# import pandas as pd
# import numpy as np

# # ─────────────────────────────────────────────
# # 🔧 설정
# # ─────────────────────────────────────────────
# base_folder = r"C:\Users\LabPC\OneDrive\주식\Financial Data\test"
# cik = "0000320193"              # Apple
# target_period = "2024Q1"        # 찾고싶은 분기 (예: 2024Q1)
# target_value = 30_736_000_000   # 기대 EBITDA
# tolerance   = 2_00_000_000      # ±200M 허용오차 (필요시 조절)

# # 태그 후보 넓게(우선순위 상→하). 회사/분기별로 쓰는 게 달라서 많이 넣음.
# TAGS = {
#     "OI":  [
#         "OperatingIncomeLoss",
#         "IncomeLossFromContinuingOperationsBeforeIncomeTaxesExtraordinaryItemsNoncontrollingInterest" # 일부가 OI로 잘못쓰는 케이스 방어
#     ],
#     "D":   [
#         "Depreciation",
#         "DepreciationExpense",
#         "DepreciationPropertyPlantAndEquipment"
#     ],
#     "A":   [
#         "Amortization",
#         "AmortizationOfIntangibleAssets",
#         "AmortizationOfPurchasedIntangibleAssets",
#         "AmortizationOfFiniteLivedIntangibleAssets"
#     ],
#     "DA_combined": [
#         "DepreciationDepletionAndAmortization",
#         "DepreciationAndAmortization",
#         "DepreciationAmortizationAndAccretionNet"
#     ],
#     "NI":  ["NetIncomeLoss","ProfitLoss","NetIncome"],
#     "TAX": ["IncomeTaxExpenseBenefit","IncomeTaxExpenseBenefitContinuingOperations"],
#     "INT": [
#         "InterestAndDebtExpense",
#         "InterestExpense",
#         "InterestExpenseDebt",
#         "InterestExpenseBorrowings",
#         "InterestAndDebtExpenseNet",
#         "InterestIncomeExpenseNet"  # 순이자(음수일 수 있음)
#     ],
# }

# # qtrs 허용(우선순위: 1→2→3→4). 다 허용해서 후처리로 분기화.
# QPRS = [1,2,3,4]

# # ─────────────────────────────────────────────
# # 🔧 유틸
# # ─────────────────────────────────────────────
# def load_all_num(base_folder):
#     quarters = sorted([f for f in os.listdir(base_folder)
#                        if f.startswith("20") and os.path.isdir(os.path.join(base_folder, f))])
#     dfs = []
#     for q in quarters:
#         p = os.path.join(base_folder, q, "num.txt")
#         if not os.path.exists(p):
#             continue
#         try:
#             df = pd.read_csv(p, sep="\t", low_memory=False)
#         except Exception:
#             continue
#         if "ddate" not in df.columns:
#             continue
#         df["value"]  = pd.to_numeric(df.get("value"), errors="coerce")
#         df["qtrs"]   = pd.to_numeric(df.get("qtrs"), errors="coerce")
#         df["period"] = pd.to_datetime(df["ddate"], format="%Y%m%d", errors="coerce")
#         df = df.dropna(subset=["period"]).copy()
#         df["Period"] = df["period"].dt.to_period("Q").astype(str)  # 예: 2024Q1
#         df["source"] = os.path.join(q, "num.txt")
#         dfs.append(df)
#     return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

# def quarterize_from_ytd(df):
#     """
#     같은 연도 내에서 qtrs>1(YTD)은 차분하여 분기값으로 환산.
#     Period, qtrs, value 컬럼 필요.
#     """
#     if df.empty:
#         return df
#     t = df.copy()
#     t["Year"] = t["Period"].str[:4].astype(int)
#     t["Q"] = t["Period"].str.extract(r"Q(\d)").astype(int)
#     out = []
#     for y, g in t.sort_values(["Year","Q","qtrs"]).groupby("Year"):
#         # 같은 분기에 qtrs가 여러개면 qtrs가 작은(=더 순수한)걸 먼저 본다
#         g = g.sort_values(["Period","qtrs"])
#         # Period별 하나 선택(가장 우선되는 qtrs)
#         g = g.groupby("Period", as_index=False).first()
#         cum = 0.0
#         last_ytd = np.nan
#         for _, r in g.iterrows():
#             val = r["value"]
#             if pd.isna(val):
#                 qval = np.nan
#             else:
#                 if int(r["qtrs"]) == 1:
#                     qval = val
#                     cum += qval
#                 else:
#                     # 누적값에서 이전까지의 분기합 차감
#                     qval = val - cum if pd.notna(cum) else val - (last_ytd if pd.notna(last_ytd) else 0)
#                     cum += (qval if pd.notna(qval) else 0)
#                     last_ytd = val
#             out.append([r["Period"], r["qtrs"], r["value"], qval, r["tag"] if "tag" in r else np.nan, r.get("source", np.nan)])
#     out_df = pd.DataFrame(out, columns=["Period","qtrs","raw_value","quarter_value","tag","source"])
#     # Period 중복시 하나만 (qtrs 작은 것 우선)
#     out_df = (out_df.sort_values(["Period","qtrs"])
#                     .groupby("Period", as_index=False).first())
#     return out_df

# def gather_metric_series(df_all, cik, tag_list):
#     """
#     특정 태그 리스트에 대해, 각 태그별로 (분기화된) Series와 메타 반환.
#     {tag: (series, meta_df)}
#     """
#     result = {}
#     for tag in tag_list:
#         sub = df_all[(df_all["adsh"].str.startswith(cik)) &
#                      (df_all["tag"]==tag) &
#                      (df_all["qtrs"].isin(QPRS)) &
#                      (df_all["segments"].isna())][["Period","qtrs","value","tag","source"]].copy()
#         if sub.empty:
#             continue
#         qdf = quarterize_from_ytd(sub)
#         s = qdf.set_index("Period")["quarter_value"]
#         result[tag] = (s, qdf)
#     return result

# def best_match(pairs, target, tol):
#     """
#     pairs: list of dict { 'name': str, 'value': float, 'detail': ... }
#     target과 가장 가까운 순으로 정렬해서 반환
#     """
#     df = pd.DataFrame([{"name": p["name"], "value": p["value"], "diff": abs(p["value"]-target), "detail": p["detail"]} for p in pairs if pd.notna(p["value"])])
#     if df.empty:
#         return df
#     return df.sort_values("diff").reset_index(drop=True)

# # ─────────────────────────────────────────────
# # 📦 메인
# # ─────────────────────────────────────────────
# num_all = load_all_num(base_folder)
# if num_all.empty:
#     raise SystemExit("num.txt 데이터를 못 찾았어요.")

# df_co = num_all[num_all["adsh"].str.startswith(cik)].copy()

# # 태그별 모든 후보 시리즈 수집 (분기화 완료된 값)
# oi_map  = gather_metric_series(df_co, cik, TAGS["OI"])
# d_map   = gather_metric_series(df_co, cik, TAGS["D"])
# a_map   = gather_metric_series(df_co, cik, TAGS["A"])
# da_map  = gather_metric_series(df_co, cik, TAGS["DA_combined"])
# ni_map  = gather_metric_series(df_co, cik, TAGS["NI"])
# tx_map  = gather_metric_series(df_co, cik, TAGS["TAX"])
# int_map = gather_metric_series(df_co, cik, TAGS["INT"])

# # 1) EBITDA = OI + (D + A)  조합들
# pairs_oi_da = []

# # (a) OI + DA_combined
# for oi_tag, (oi_s, oi_meta) in oi_map.items():
#     for da_tag, (da_s, da_meta) in da_map.items():
#         v = (oi_s.get(target_period, np.nan)) + (da_s.get(target_period, np.nan))
#         pairs_oi_da.append({
#             "name": f"{oi_tag} + {da_tag}",
#             "value": v,
#             "detail": {
#                 "OI": {"tag": oi_tag, "meta": oi_meta[oi_meta["Period"]==target_period].to_dict("records")},
#                 "DA": {"tag": da_tag, "meta": da_meta[da_meta["Period"]==target_period].to_dict("records")},
#             }
#         })

# # (b) OI + (D + A) (분리계정 합산)
# for oi_tag, (oi_s, oi_meta) in oi_map.items():
#     for d_tag, (d_s, d_meta) in d_map.items():
#         for a_tag, (a_s, a_meta) in a_map.items():
#             v = (oi_s.get(target_period, np.nan)) + (d_s.get(target_period, np.nan)) + (a_s.get(target_period, np.nan))
#             pairs_oi_da.append({
#                 "name": f"{oi_tag} + {d_tag} + {a_tag}",
#                 "value": v,
#                 "detail": {
#                     "OI": {"tag": oi_tag, "meta": oi_meta[oi_meta["Period"]==target_period].to_dict("records")},
#                     "D":  {"tag": d_tag,  "meta": d_meta[d_meta["Period"]==target_period].to_dict("records")},
#                     "A":  {"tag": a_tag,  "meta": a_meta[a_meta["Period"]==target_period].to_dict("records")},
#                 }
#             })

# best_oi_da = best_match(pairs_oi_da, target_value, tolerance)

# # 2) EBITDA = NI + TAX + INT + (D + A)
# pairs_ni_ti_da = []

# # (a) NI + TAX + INT + DA_combined
# for ni_tag, (ni_s, ni_meta) in ni_map.items():
#     for tx_tag, (tx_s, tx_meta) in tx_map.items():
#         for it_tag, (it_s, it_meta) in int_map.items():
#             for da_tag, (da_s, da_meta) in da_map.items():
#                 v = ni_s.get(target_period, np.nan) + tx_s.get(target_period, np.nan) + it_s.get(target_period, np.nan) + da_s.get(target_period, np.nan)
#                 pairs_ni_ti_da.append({
#                     "name": f"{ni_tag} + {tx_tag} + {it_tag} + {da_tag}",
#                     "value": v,
#                     "detail": {
#                         "NI": {"tag": ni_tag, "meta": ni_meta[ni_meta["Period"]==target_period].to_dict("records")},
#                         "TAX":{"tag": tx_tag, "meta": tx_meta[tx_meta["Period"]==target_period].to_dict("records")},
#                         "INT":{"tag": it_tag, "meta": it_meta[it_meta["Period"]==target_period].to_dict("records")},
#                         "DA": {"tag": da_tag, "meta": da_meta[da_meta["Period"]==target_period].to_dict("records")},
#                     }
#                 })

# # (b) NI + TAX + INT + (D + A) (분리계정 합산)
# for ni_tag, (ni_s, ni_meta) in ni_map.items():
#     for tx_tag, (tx_s, tx_meta) in tx_map.items():
#         for it_tag, (it_s, it_meta) in int_map.items():
#             for d_tag, (d_s, d_meta) in d_map.items():
#                 for a_tag, (a_s, a_meta) in a_map.items():
#                     v = ni_s.get(target_period, np.nan) + tx_s.get(target_period, np.nan) + it_s.get(target_period, np.nan) + d_s.get(target_period, np.nan) + a_s.get(target_period, np.nan)
#                     pairs_ni_ti_da.append({
#                         "name": f"{ni_tag} + {tx_tag} + {it_tag} + {d_tag} + {a_tag}",
#                         "value": v,
#                         "detail": {
#                             "NI": {"tag": ni_tag, "meta": ni_meta[ni_meta["Period"]==target_period].to_dict("records")},
#                             "TAX":{"tag": tx_tag, "meta": tx_meta[tx_meta["Period"]==target_period].to_dict("records")},
#                             "INT":{"tag": it_tag, "meta": it_meta[it_meta["Period"]==target_period].to_dict("records")},
#                             "D":  {"tag": d_tag,  "meta": d_meta[d_meta["Period"]==target_period].to_dict("records")},
#                             "A":  {"tag": a_tag,  "meta": a_meta[a_meta["Period"]==target_period].to_dict("records")},
#                         }
#                     })

# best_ni_ti_da = best_match(pairs_ni_ti_da, target_value, tolerance)

# # ─────────────────────────────────────────────
# # 🖨 결과
# # ─────────────────────────────────────────────
# print(f"🎯 Target: {target_period} → {target_value:,} (±{tolerance:,})")
# printed = False

# def print_hits(df, title):
#     global printed
#     if df is not None and not df.empty and df.loc[0, "diff"] <= tolerance:
#         print(f"\n✅ Matches ({title})")
#         for i, row in df[df["diff"]<=tolerance].head(10).iterrows():
#             print(f"- {row['name']}: {int(row['value']):,}  (diff={int(row['diff']):,})")
#             print("  detail:", row["detail"])
#             printed = True

# print_hits(best_oi_da, "EBITDA = OI + D&A")
# print_hits(best_ni_ti_da, "EBITDA = NI + TAX + INT + D&A")

# if not printed:
#     # 근접값 TOP 10 힌트
#     print("\n❌ 정확 일치 없음. 가장 가까운 조합 Top 10 (참고):")
#     near = pd.concat([
#         best_oi_da.assign(group="OI+D&A").head(10),
#         best_ni_ti_da.assign(group="NI+TAX+INT+D&A").head(10)
#     ], ignore_index=True)
#     if not near.empty:
#         near = near.sort_values("diff").head(10)
#         for i, row in near.iterrows():
#             print(f"- [{row['group']}] {row['name']}: {int(row['value']):,} (diff={int(row['diff']):,})")
#             # 원천 메타는 너무 길 수 있어 필요시 주석 해제
#             # print("  detail:", row["detail"])
#     else:
#         print("  후보 시리즈가 부족합니다. 태그 후보를 더 추가하세요.")
